In [4]:
from dotenv import load_dotenv
import os, json
from openai import OpenAI

load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")
default_model = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-6-luna")
google_api_key = os.getenv("GOOGLE_API_KEY")

client = OpenAI()


In [3]:
from openai import OpenAI

client = OpenAI()
res = client.responses.create(
    model = default_model,
    input = '오늘 서울 날시 알려줘',
    tools = [{'type':'web_search'}]
)

print(res.output_text)

오늘(11월 18일) 서울은 **흐리고**, 현재 약 **9°C**예요. 오후에는 **소나기 가능성**이 있고, 최고 **10°C**, 최저 **-1°C**로 쌀쌀하니 외출하실 때 따뜻하게 입으세요. 


In [9]:
vector_store = client.vector_stores.create(name='호텔 매뉴얼')

In [10]:
vector_store

VectorStore(id='vs_6afcf3ab89e08191ae14da9f316be8ac', created_at=1794962347, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0), last_active_at=1794962347, metadata={}, name='호텔 매뉴얼', object='vector_store', status='completed', usage_bytes=0, expires_after=None, expires_at=None, description=None)

In [ ]:
print(r'\%')

\%


<>:1: SyntaxWarning: invalid escape sequence '\%'
<>:1: SyntaxWarning: invalid escape sequence '\%'
C:\Users\Admin\AppData\Local\Temp\ipykernel_15568\3949692331.py:1: SyntaxWarning: invalid escape sequence '\%'
  print('\%')


In [17]:
pdf_file = r'..\..\lecture\data\Hotel Service Manual.pdf'

vector_store = client.vector_stores.create(name='호텔 매뉴얼')

with open(pdf_file, 'rb') as f:
    file_batch = client.vector_stores.file_batches.upload_and_poll(
        vector_store_id= vector_store.id,
        files = [f]
    )


In [ ]:
print(file_batch.status, file_batch.file_counts)
res = client.responses.create(
    model = default_model,
    input = '호텔 메뉴얼 내용 알려줘',
    tools = [{'type':'file_search', 'vector_store_ids':[vector_store.id]}]
)
print(res.output_text)

업로드된 **호텔 고객 응대 서비스 매뉴얼**은 프런트·객실팀 직원이 고객에게 일관된 서비스를 제공하기 위한 표준 안내서입니다. 주요 내용은 다음과 같습니다.

- **손님맞이:** 미소와 예의 있는 인사로 맞이하고, 고객 이름을 알면 개인화해 인사합니다. 
- **체크인·체크아웃:** 체크인은 15시, 체크아웃은 11시입니다. 조기 체크인이나 늦은 체크아웃은 객실 상황에 따라 안내하며, 어려운 경우 짐 보관을 제안합니다. 
- **Wi‑Fi·주차:** 객실 무료 Wi‑Fi와 180대 규모의 무료 주차장을 안내합니다. 연결 방법, 주차 위치와 이용 방법도 설명해야 합니다. 
- **배리어 프리·반려동물:** 유니버설 룸과 장애인 편의시설을 안내하고 필요한 도움을 제공합니다. 반려동물 동반은 불가하므로 정중히 알리고, 인근 동반 가능 숙소를 안내합니다. 
- **룸서비스·흡연:** 룸서비스는 23시까지 운영하며, 알레르기나 식단 제한은 주방과 협력해 대응합니다. 객실은 모두 금연이고, 1층 흡연실을 안내합니다. 
- **취소·결제:** 전날 취소는 숙박요금의 30%, 당일 취소는 50%, 노쇼는 100%의 수수료가 발생합니다. 결제는 프런트에서 현금·신용카드·직불카드로 할 수 있고, 온라인 예약 시 카드 결제도 가능합니다. 
- **기본 원칙:** 모든 고객을 존중하고 예의와 배려, 전문성을 갖춰 응대합니다. 


In [18]:
client.vector_stores.delete(vector_store.id)

VectorStoreDeleted(id='vs_6afcf6dea6608191802e6873a748b668', deleted=True, object='vector_store.deleted')

In [20]:
# 내가 만든 함수(tool)을 사용해서 아래 질문에 답을 구할 예정
# 오늘 서울 날씨 어때? 거기에 10을 곱해줘

response = client.responses.create(
    model = default_model,
    input = "오늘 날씨가 뭐야?"
)


In [19]:
def get_weather(location: str) -> str:
    """지정한 도시의 날씨 정보를 반환합니다. (실습용 고정 데이터)"""
    weather_data = {
        "서울": "맑고 기온은 24도입니다.",
        "뉴욕": "비가 오고 있으며 기온은 15도입니다.",
        "도쿄": "흐림, 기온은 20도입니다.",
    }
    return weather_data.get(location, f"{location}의 날씨 정보가 없습니다.")

print(get_weather("서울"))
print(get_weather("파리"))

맑고 기온은 24도입니다.
파리의 날씨 정보가 없습니다.


In [30]:
def multi(a:int,b:int)->int:
    '''두 수를 곱한 결과를 반환합니다.'''
    return a * b

In [36]:
weather_tools = [{
    "type": "function",
    "name": "get_weather",
    "description": "지정한 도시의 현재 날씨를 조회한다.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "날씨를 조회할 도시 이름. 예: 서울, 뉴욕, 도쿄",
            }
        },
        "required": ["location"],
        "additionalProperties": False,
    },
    "strict": True,
},
                {
    "type": "function",
        "name": "multi",
        "strict": True,
        "description": "두 수를 곱한 결과를 반환합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "integer",
                    "description": "첫번째 숫자",
                },
                "b": {
                    "type": "integer",
                    "description": "두번째 숫자"
                } 
            },
            "required": ["a", "b"],
            "additionalProperties": False,
        },
        "strict": True,

}]

In [32]:
tool_map = {"get_weather": get_weather, "multi": multi }

In [ ]:
from types import SimpleNamespace

# 모델이 이런 응답을 보냈다고 가정 (실제로는 resp1.output 안에 들어 있음)
fake_output = [
    SimpleNamespace(type="reasoning"),                          # 추론 모델은 이 항목이 먼저 올 수 있음
    SimpleNamespace(type="function_call", name="get_weather",
                    arguments='{"location": "서울"}', call_id="call_123"),
]
tool_outputs = []
for item in fake_output:
    if item.type != "function_call":          # 함수 호출이 아닌 항목은 건너뜀
        continue
    args = json.loads(item.arguments)         # JSON 문자열 → 딕셔너리
    result = tool_map[item.name](**args)      # 이름으로 함수를 찾아 실행
    tool_outputs.append({
        "type": "function_call_output",
        "call_id": item.call_id,              # 어떤 요청에 대한 결과인지 짝을 맞춤
        "output": result,                     # 문자열이어야 함 (dict는 json.dumps)
    })

print(json.dumps(tool_outputs, ensure_ascii=False, indent=2))

[
  {
    "type": "function_call_output",
    "call_id": "call_123",
    "output": "맑고 기온은 24도입니다."
  }
]


In [33]:
input_message = "서울의 날씨는 어때?"
# input_message = "바나나는 무슨 색깔이니?"      # 도구가 필요 없는 질문으로도 시험해 보기

resp1 = client.responses.create(
    model=default_model,
    input=input_message,
    tools=weather_tools,
    tool_choice="auto",          # 필요하면 도구 호출, 아니면 바로 답변
)

for item in resp1.output:        # output 항목을 모두 확인
    print(item.type, end="  ")
    if item.type == "function_call":
        print("→", item.name, item.arguments, item.call_id)
    else:
        print()

function_call  → get_weather {"location":"서울"} call_RYJUNkYu92i7EIfKC9nTsCgd


In [34]:
# 1) 모델이 요청한 함수를 실행
tool_outputs = []
for item in resp1.output:
    if item.type != "function_call":
        continue
    args = json.loads(item.arguments)
    result = tool_map[item.name](**args)
    tool_outputs.append({"type": "function_call_output", "call_id": item.call_id, "output": result})

# 2) 도구가 필요 없었으면 바로 답변, 필요했으면 결과를 보내 최종 답변
if not tool_outputs:
    print(resp1.output_text)
else:
    resp2 = client.responses.create(
        model=default_model,
        previous_response_id=resp1.id,     # 1차 요청 대화에 이어서
        input=tool_outputs,                # 함수 실행 결과 전달
        tools=weather_tools,
    )
    print(resp2.output_text)

서울은 현재 맑고, 기온은 24도예요.


In [35]:
def ask_weather(question: str) -> str:
    resp = client.responses.create(model=default_model, input=question,
                                   tools=weather_tools, tool_choice="auto")
    outputs = []
    for item in resp.output:
        if item.type == "function_call":
            result = tool_map[item.name](**json.loads(item.arguments))
            outputs.append({"type": "function_call_output", "call_id": item.call_id, "output": result})
    if not outputs:
        return resp.output_text
    final = client.responses.create(model=default_model, previous_response_id=resp.id,
                                    input=outputs, tools=weather_tools)
    return final.output_text

print(ask_weather("서울의 날씨는 어때?"))
print(ask_weather("바나나는 무슨 색깔이니?"))

서울은 현재 맑고, 기온은 24도예요.
바나나는 보통 노란색이에요. 덜 익으면 초록색이고, 아주 익으면 갈색 반점이 생길 수 있어요.


In [29]:
def run_with_tools(user_input: str, tools: list, tool_map: dict) -> str:
    """모델의 함수 호출 요청을 모두 처리한 뒤 최종 답변을 반환합니다."""
    response = client.responses.create(
        model=default_model, input=user_input, tools=tools, tool_choice="auto"
    )

    while True:
        tool_outputs = []
        for item in response.output:
            if item.type != "function_call":
                continue
            try:
                arguments = json.loads(item.arguments)
                result = tool_map[item.name](**arguments)
            except Exception as error:
                result = {"error": f"도구 실행 중 오류: {error}"}

            tool_outputs.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps(result, ensure_ascii=False)
                if isinstance(result, dict) else str(result),
            })

        if not tool_outputs:
            return response.output_text

        response = client.responses.create(
            model=default_model,
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools,
        )

print(run_with_tools("서울의 날씨는 어떠니? 서울 날씨에 10을 곱해줘", weather_tools, tool_map))


서울은 맑고 기온은 24도예요. 기온에 10을 곱하면 **240도**입니다.


In [3]:
import json
from forex_python.converter import CurrencyRates

def get_exchange_rate(current:str, cg_current:str):
    '''특정 통화의 환율 정보를 알려준다.'''
    c = CurrencyRates()
    rate = c.get_rate({current}, {cg_current})
    return round(rate,2)

def divide_ab(a:int, b:int)->int:
    '''두 수를 나눈 결과를 반환합니다.'''
    if b == 0:
        return  
    return a / b


def substract(a:int, b:int)->int:
    '''두 수를 뺀 결과를 반환합니다.'''
    return a - b

def add_ab(a:int, b:int)->int:
    '''두 수를 더한 결과를 반환합니다.'''
    return a + b

def multi(a:int,b:int)->int:
    '''두 수를 곱한 결과를 반환합니다.'''
    return a * b

def get_weather(location:str)-> str:
    """지정한 도시의 날씨 정보를 반환합니다 """
    weather_data = {
        "서울": "맑고 기온은 24도입니다.",
        "뉴욕": "비가 오고 있으며 기온은 15도입니다.",
        "도쿄": "흐림, 기온은 20도입니다.",
    }
    ret = weather_data.get(location, '해당 도시의 날씨 정보를 찾을 수 없습니다.')
    return ret


weather_tools = [
    {
        "type": "function",
        "name": "get_weather",
        "strict": True,
        "description": "지정한 도시의 날씨 정보를 반환합니다",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "날씨 정보를 얻고자 하는 도시 이름",
                },
            },
            "required": ["location"],
            "additionalProperties": False,
        },
        'strict': True
    },
    {
        "type": "function",
        "name": "multi",
        "strict": True,
        "description": "두 수를 곱한 결과를 반환합니다",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "integer",
                    "description": "첫 번째 숫자",
                },
                "b": {
                    "type": "integer",
                    "description": "두 번째 숫자",
                },
            },
            "required": ["a", "b"],
            "additionalProperties": False,
        },
        'strict': True
    },    
]


tool_map = {'get_weather': get_weather, 'multi':multi}

def run_with_tools(user_input: str, tools: list, tool_map: dict) -> str:
    """모델의 함수 호출 요청을 모두 처리한 뒤 최종 답변을 반환합니다."""
    response = client.responses.create(
        model=default_model, input=user_input, tools=tools, tool_choice="auto"
    )

    while True:
        tool_outputs = []
        for item in response.output:
            if item.type != "function_call":
                continue
            try:
                arguments = json.loads(item.arguments)
                result = tool_map[item.name](**arguments)
            except Exception as error:
                result = {"error": f"도구 실행 중 오류: {error}"}

            tool_outputs.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps(result, ensure_ascii=False)
                if isinstance(result, dict) else str(result),
            })
            print(f"Function call: {item.name}, Arguments: {arguments}, Result: {result}")

        if not tool_outputs:
            return response.output_text

        response = client.responses.create(
            model=default_model,
            instructions = '툴이 있으면 반드시 내 툴을 이용해서 결과를 알려줘',
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools,
        )

print(run_with_tools("서울의 날씨는 어떠니? 서울 날씨에 10을 곱해줘", weather_tools, tool_map))

NameError: name 'client' is not defined

In [ ]:
import json
from openai import OpenAI
from forex_python.converter import CurrencyRates, RatesNotAvailableError

# OpenAI 클라이언트 및 모델 설정 (환경변수 OPENAI_API_KEY 필요)
client = OpenAI()
default_model = "gpt-4o-mini"


# 1. 사칙연산 함수 4개
def add_ab(a: float, b: float) -> float:
    """두 수를 더한 결과를 반환합니다."""
    return a + b


def substract(a: float, b: float) -> float:
    """첫 번째 수에서 두 번째 수를 뺀 결과를 반환합니다."""
    return a - b


def multi(a: float, b: float) -> float:
    """두 수를 곱한 결과를 반환합니다."""
    return a * b


def divide_ab(a: float, b: float) -> float | str:
    """첫 번째 수를 두 번째 수로 나눈 결과를 반환합니다. 0으로 나누는 경우 예외 처리합니다."""
    if b == 0:
        return "오류: 0으로 나눌 수 없습니다."
    return a / b


# 2. 날씨 조회 함수
def get_weather(location: str) -> str:
    """지정한 도시의 날씨 정보를 반환합니다."""
    # '제주도'로 입력해도 '제주'와 매칭되도록 전처리
    normalized_loc = location.replace("제주도", "제주").strip()

    weather_data = {
        "서울": "맑고 기온은 24도입니다.",
        "부산": "구름 조금, 기온은 22도입니다.",
        "제주": "바람이 불고 맑으며 기온은 21도입니다.",
        "도쿄": "흐림, 기온은 20도입니다.",
        "뉴욕": "비가 오고 있으며 기온은 15도입니다.",
    }
    return weather_data.get(
        normalized_loc, f"'{location}'의 날씨 정보를 찾을 수 없습니다."
    )


# 3. 환율 조회 함수 (원화 KRW 기준)
def get_exchange_rate(currency_code: str) -> str:
    """입력받은 통화 코드(USD, JPY, EUR 등)의 원화(KRW) 기준 실제 환율 정보를 반환합니다."""
    code = currency_code.upper().strip()

    try:
        c = CurrencyRates()
        rate = c.get_rate(code, "KRW")
        formatted_rate = f"{round(rate, 2):,}"
        return f"1 {code}는 약 {formatted_rate}원입니다."
    except RatesNotAvailableError:
        return f"지원하지 않거나 현재 조회할 수 없는 통화 코드입니다: {currency_code}"
    except Exception as e:
        return f"환율 API 호출 중 오류가 발생했습니다: {e}"


# OpenAI API tools에 6개 함수 모두 등록
tools = [
    {
        "type": "function",
        "name": "add_ab",
        "strict": True,
        "description": "두 수를 더한 결과를 반환합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "number", "description": "첫 번째 숫자"},
                "b": {"type": "number", "description": "두 번째 숫자"},
            },
            "required": ["a", "b"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "substract",
        "strict": True,
        "description": "첫 번째 수(a)에서 두 번째 수(b)를 뺀 결과를 반환합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "number", "description": "첫 번째 숫자"},
                "b": {"type": "number", "description": "뺄 숫자"},
            },
            "required": ["a", "b"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "multi",
        "strict": True,
        "description": "두 수를 곱한 결과를 반환합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "number", "description": "첫 번째 숫자"},
                "b": {"type": "number", "description": "두 번째 숫자"},
            },
            "required": ["a", "b"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "divide_ab",
        "strict": True,
        "description": "첫 번째 수(a)를 두 번째 수(b)로 나눈 결과를 반환합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "number", "description": "나누어지는 수"},
                "b": {"type": "number", "description": "나누는 수"},
            },
            "required": ["a", "b"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "get_weather",
        "strict": True,
        "description": "지정한 도시의 날씨 정보를 반환합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "날씨를 조회할 도시 이름 (예: 서울, 부산, 제주, 도쿄)",
                },
            },
            "required": ["location"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "get_exchange_rate",
        "strict": True,
        "description": "기준 통화 코드(USD, JPY, EUR 등)의 원화(KRW) 기준 환율 정보를 알려줍니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "currency_code": {
                    "type": "string",
                    "description": "조회할 통화 코드 (예: USD, JPY, EUR)",
                },
            },
            "required": ["currency_code"],
            "additionalProperties": False,
        },
    },
]

# 함수 이름과 실제 Python 함수 매핑
tool_map = {
    "add_ab": add_ab,
    "substract": substract,
    "multi": multi,
    "divide_ab": divide_ab,
    "get_weather": get_weather,
    "get_exchange_rate": get_exchange_rate,
}


def run_with_tools(user_input: str) -> str:
    """모델의 함수 호출 요청을 처리하고 최종 자연어 답변을 반환합니다."""
    system_instruction = (
        "제공된 도구(tools)를 활용하여 사용자의 질문에 정확히 답하세요. "
        "환전 금액 계산이 필요한 경우 먼저 환율 조회 도구로 환율을 확인한 뒤 곱셈 도구를 사용하세요."
    )

    response = client.responses.create(
        model=default_model,
        instructions=system_instruction,
        input=user_input,
        tools=tools,
        tool_choice="auto",
    )

    while True:
        tool_outputs = []
        for item in response.output:
            if item.type != "function_call":
                continue

            # 1. 함수 이름과 인자 확인
            func_name = item.name
            arguments = json.loads(item.arguments)

            # 2. 실제 Python 함수 실행
            try:
                result = tool_map[func_name](**arguments)
            except Exception as error:
                result = f"도구 실행 중 오류 발생: {error}"

            print(f"  [Tool Call] {func_name}({arguments}) -> {result}")

            # 3. 실행 결과를 function_call_output으로 구성
            tool_outputs.append(
                {
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": str(result),
                }
            )

        # 더 이상 호출할 함수가 없으면 최종 자연어 답변 반환
        if not tool_outputs:
            return response.output_text

        # 4. 결과를 다시 모델에 전달
        response = client.responses.create(
            model=default_model,
            instructions=system_instruction,
            previous_response_id=response.id,
            input=tool_outputs,
            tools=tools,
        )


# 테스트 실행 (최소 5개 이상 다양한 유형 검증)
test_questions = [
    "10을 0으로 나눠줘.",
    "제주도 날씨는 어때?",
    "부산 날씨를 알려주고, 25 곱하기 4도 계산해줘.",
    "오늘 달러 환율 알려줘.",
    "서울 날씨와 엔화 환율을 함께 알려줘.",
    "100달러를 원화로 계산해줘.",
]

if __name__ == "__main__":
    for q in test_questions:
        print(f"\n질문: {q}")
        answer = run_with_tools(q)
        print(f"답변: {answer}")